<a href="https://colab.research.google.com/github/Dharmendra0305/PRODIGY_GA_01/blob/main/Text_Generation_with_GPT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phase 1: Environment Setup**

* Check GPU Availability



In [ ]:
import torch
torch.cuda.is_available()

* Install Core Libraries

In [ ]:
!pip install transformers datasets torch

* Verify Installation

In [ ]:
import transformers
import datasets
import torch

print("Transformers Version:", transformers.__version__)
print("Datasets Version:", datasets.__version__)
print("PyTorch Version:", torch.__version__)

# **Phase 2: Data Preparation**

* Collect Text

In [ ]:
from google.colab import files

uploaded = files.upload()

* Load Text

In [ ]:
with open("gpt.txt", "r", encoding = "utf-8") as f:
  text = f.read()

print(text[:500])

* Tokenization

In [ ]:
from transformers import AutoTokenizer

# Load GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Assign EOS token as pad token
tokenizer.pad_token = tokenizer.eos_token

# Tokenize text
tokens = tokenizer(text, return_tensors = "pt")

print(tokens["input_ids"][0][:50])

* Chunking

In [ ]:
block_size = 128

# Convert text into token IDs
token_ids = tokenizer.encode(text)

# Break into chunks
chunks = [token_ids[i:i + block_size] for i in range(0, len(token_ids), block_size)]

print("Number of chunks:", len(chunks))
print("First chunk:", chunks[0])

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({"input_ids": chunks})
print(dataset)

# **Phase 3: Fine-Tuning (Training)**



* Load the Base Model

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")

* Prepare the Dataset

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer,
                                                mlm = False)           # causal LM (GPT-2) does not use masked language modeling

* Configure the Trainer

In [ ]:
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir = "./results",        # where to save checkpoints

    num_train_epochs = 3,            # train for 3 cycles

    per_device_train_batch_size = 2, # batch size per GPU

    save_steps = 500,                # save every 500 steps

    save_total_limit = 2,            # keep only last 2 checkpoints

    logging_steps = 100              # log every 100 steps
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = dataset,
    data_collator = data_collator
)

* Execute Training

In [ ]:
trainer.train()

* Save the Model

In [ ]:
trainer.save_model("./fine_tuned_gpt2")

# **Phase 4: Generation & Evaluation**

* Load Fine-Tuned Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token         # Keep the padding fix

* Prepare a Prompt

In [ ]:
prompt = "That empire was given to Rome not by the"

* Tokenize the Prompt

In [ ]:
inputs = tokenizer(prompt, return_tensors = "pt", padding = True, return_attention_mask = True)

* Generate Text

In [ ]:
outputs = model.generate(
    inputs["input_ids"],
    attention_mask = inputs["attention_mask"],

    pad_token_id = tokenizer.eos_token_id,  # explicitly set pad token

    max_length = 150,                       # total length of generated text

    num_return_sequences = 1,               # how many completions to generate

    temperature = 0.9,                      # creativity (lower = more focused, higher = more random)

    top_p = 0.85,                           # nucleus sampling (controls diversity)

    repetition_penalty = 1.2,               # discourages loops

    do_sample = True                        # enables sampling instead of greedy decoding
)

print(tokenizer.decode(outputs[0], skip_special_tokens = True))

# **Phase 5: Interactive Web App**

In [ ]:
import gradio as gr

# 1. Load your trained model and tokenizer
model_path = "./custom-gpt2-final"
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# 2. Define the function that runs when a user clicks "Submit"
def generate_custom_text(prompt, temperature, max_length):
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        **inputs,
        max_length=max_length,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# 3. Build the user interface
app = gr.Interface(
    fn=generate_custom_text,
    inputs=[
        gr.Textbox(lines=3, label="Starting Prompt", placeholder="Type here..."),
        gr.Slider(minimum=0.1, maximum=1.5, value=0.7, label="Temperature (Creativity)"),
        gr.Slider(minimum=10, maximum=200, step=10, value=50, label="Max Length")
    ],
    outputs=gr.Textbox(label="Generated Output"),
    title="Custom GPT-2 Text Generator"
)

# 4. Launch the web app
app.launch(share=True)